# Mk33: does memory change task success? (Kaggle)

The earlier memory notebooks measured **next-token loss**. This one measures what matters for an agent:
**does it solve more of the task?** The Mk33 agent runs end to end on the bench tasks with each memory method, at an
equal memory budget, and the report compares final scores with paired bootstrap 95% intervals.

| mode | how old turns are forgotten |
| --- | --- |
| `truncate` | oldest turns are cut from the text (baseline) |
| `rlc` | old turns are replaced by the round ledger |
| `retrieve` | old turns go into BM25 retrieval memory and relevant ones come back |
| `kv:full` | nothing is forgotten (upper bound, uses 4x the memory) |
| `kv:h2o` | the model keeps the full text; its KV cache is cut to the same budget by H2O eviction |

**Model.** Qwen2.5-7B-Instruct with 4-bit weights (about 5 GB), so it fits one T4 with room for the KV cache. An
earlier grid with Qwen2.5-3B failed for reasons unrelated to memory in 5 of 6 task-repeats (broken code, no
submissions), so it could not compare memory methods. Only the weights are quantized; the KV cache and the eviction
policies run in float16 as before.

**Time.** 30 agent runs of up to 25 minutes, usually one or two Kaggle sessions. Finished runs are never repeated.

## How to run on Kaggle

1. Kaggle account with a **verified phone number** (Settings > Phone verification); needed for GPUs and internet.
2. In this notebook's right-hand panel: **Accelerator: GPU T4 x2** (one GPU is used) and **Internet: on**.
3. Click **Save Version**, choose **Save & Run All (Commit)**, and save. Kaggle runs the whole notebook in the background for up to 12 hours; you can close the browser.
4. When it finishes, the results are in the version's **Output** tab, and the report is printed near the end of the log.

**Continuing in a new session.** The notebook stops starting new runs about an hour before Kaggle's 12-hour limit, so every finished run is saved. To continue: **Add Input** (right panel) > **Your Work** > this notebook, pick the previous version, then Save Version again. The notebook copies the earlier results in first and skips everything already finished.

Kaggle gives about 30 free GPU hours per week; the remaining quota is shown in Settings.

## 1. GPU and model

In [ ]:
import time, os
START = time.time()
SESSION_HOURS = 11.0   # Kaggle stops commits at 12 h; leave a margin so the output is saved


def hours_left():
    return SESSION_HOURS - (time.time() - START) / 3600


def have_time(hours_needed):
    """True if there is time to start a step that may take up to `hours_needed` hours."""
    ok = hours_left() >= hours_needed
    if not ok:
        print(f"Stopping here to save results ({hours_left():.1f} h left in this session). "
              "Continue in a new version with this version's output added as input.")
    return ok

import torch
assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU T4 x2 in the right panel."
mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name()}, {mem_gb:.0f} GB")

MODEL = "Qwen/Qwen2.5-7B-Instruct"  # 3B failed too often to compare memory methods (see the top of the notebook)
QUANT = "4bit"                       # weights only, about 5 GB; set "" for full precision on a larger GPU
DTYPE = "float16"                    # T4 has no fast bfloat16

## 2. Results folder

Everything goes to `/kaggle/working/task_success_7b`, which becomes this version's output.

In [ ]:
import glob, shutil
OUT_DIR = "/kaggle/working/task_success_7b"
# Results from an earlier version of this notebook, attached with Add Input > Your Work.
previous = sorted(glob.glob("/kaggle/input/*/task_success_7b")) + sorted(glob.glob("/kaggle/input/*/*/task_success_7b"))
for p in previous:
    print("continuing from", p)
    shutil.copytree(p, OUT_DIR, dirs_exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("results folder:", OUT_DIR, "| finished runs so far:",
      len(glob.glob(f"{OUT_DIR}/**/result.json", recursive=True)))
OUT = OUT_DIR

## 3. Install Mk33

In [ ]:
%cd /tmp
!test -d Mk33 || git clone -q https://github.com/dulguuntsatsralt-cell/Mk33
%cd /tmp/Mk33
!git pull -q
!pip install -q -e '.[mk33-kv,dev]'
!git log --oneline -1

## 4. Prepare the tasks and check the pipeline

Task generation is seeded, so every session builds identical environments. The dry run checks the whole pipeline
(workspace copy, scorer, ledger, report) in a few seconds without loading the model.

In [ ]:
!mk33 bench-prepare --out bench/envs --tasks knapsack,regression
!mk33 bench-run --dry-run --tasks knapsack --modes rlc --repeats 1 --max-rounds 2 --out /tmp/dry --force | tail -2

## 4b. Check the model before the grid

Loads the model the way the grid will, sends the agent's real first prompt for the knapsack task with the bench's
tools, and checks that the reply is a usable tool call. If loading fails or the model does not call a tool, this
cell stops the notebook here, before hours of GPU time go into broken runs. It then frees the GPU for the grid.

In [ ]:
import gc, time, torch
from pathlib import Path
from mk33.bench.run import BENCH_TOOLS
from mk33.kv.client import HFChatClient
from mk33.scaffold.context import SYSTEM_PROMPT
from mk33.scaffold.tools import TOOL_SCHEMAS

t0 = time.time()
client = HFChatClient(MODEL, dtype=DTYPE, quantize=QUANT, max_tokens=1024)
print(f"loaded in {time.time() - t0:.0f} s; GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

task = Path("bench/envs/knapsack/workspace/README.md").read_text(encoding="utf-8")
messages = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"# Task\n{task}\n\n# Current best\nNo scored submission yet."}]
t0 = time.time()
reply = client.chat(messages, tools=[TOOL_SCHEMAS[t] for t in BENCH_TOOLS])
calls = reply["tool_calls"]
print(f"one model call: {time.time() - t0:.0f} s")
print("tool call:", calls[0]["name"] if calls else None, str(calls[0]["arguments"])[:200] if calls else "")
ok = bool(calls) and calls[0]["name"] in BENCH_TOOLS

del client; gc.collect(); torch.cuda.empty_cache()   # the grid loads its own copy
if not ok:
    raise RuntimeError("The model did not return a usable tool call, so the grid would waste GPU time. "
                       "Reply was: " + reply["content"][:500])
print("OK: the model loads in 4-bit and calls the bench tools.")

## 5. Run the grid

Settings shared by every mode:

* 2 tasks (knapsack, regression) x 3 repeats
* up to 8 rounds, 40 tool calls and 25 minutes per run
* memory budget 8,000 tokens: the text window for `truncate`, `rlc` and `retrieve`, and the KV-cache budget for `kv:h2o`
* `kv:full` keeps up to 32,000 tokens and is the no-forgetting reference

Each mode runs in its own call (H2O with a smaller prefill chunk, since it needs eager attention); all calls
write to the same folder and the report combines them.

In [ ]:
COMMON = (f"--backend hf --model {MODEL} --dtype {DTYPE} --quantize {QUANT} --envs bench/envs/envs.jsonl --out {OUT} "
          "--tasks knapsack,regression --max-rounds 8 --max-tool-calls 40 --max-minutes 25 "
          "--window 8000 --kv-budget 8000 --kv-text-window 32000 --temperature 0.7")

# One call per (repeat, mode): at most 2 runs (one per task), so at most about 50 minutes.
# Repeat 1 finishes for every mode before repeat 2 starts, so partial results stay balanced.
stopped = False
for rep in (1, 2, 3):
    for mode in ["truncate", "rlc", "retrieve", "kv:full", "kv:h2o"]:
        if not have_time(1.0):
            stopped = True
            break
        extra = "--kv-chunk 128" if mode == "kv:h2o" else ""
        print(f"=== repeat {rep}, {mode} ({hours_left():.1f} h left) ===")
        !mk33 bench-run {COMMON} --repeats {rep} --modes {mode} {extra} 2>&1 | grep -v -i warning | tail -3
    if stopped:
        break

## 6. Report

`report.md` has the tables: mean best score per mode, the share of runs that beat the reference solution, and paired
differences with 95% intervals. A difference counts as real only when its interval excludes zero.

In [ ]:
!mk33 bench-report --out {OUT}

In [ ]:
import json, statistics
import matplotlib.pyplot as plt

s = json.load(open(f"{OUT}/summary.json"))
order = [m for m in ["truncate", "rlc", "retrieve", "kv:h2o", "kv:full"] if m in s["modes"]]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

means = [statistics.mean(s["per_mode"][m]["best"]) for m in order]
for i, m in enumerate(order):
    xs = s["per_mode"][m]["best"]
    ax1.scatter([i] * len(xs), xs, color="0.6", s=18, zorder=2)
ax1.bar(range(len(order)), means, color=["#888" if m in ("truncate", "kv:full") else "#3b5bdb" for m in order], zorder=1)
ax1.axhline(1.0, ls="--", lw=1, color="k"); ax1.text(len(order) - 0.5, 1.01, "reference", ha="right", fontsize=8)
ax1.set_xticks(range(len(order)), order); ax1.set_ylabel("best normalized score"); ax1.set_title("Final score per run")

for m in order:
    c = s["curves"][m]
    ax2.plot(range(1, len(c) + 1), c, marker="o", ms=3, label=m)
ax2.set_xlabel("round"); ax2.set_ylabel("best so far (mean)"); ax2.set_title("Progress over rounds"); ax2.legend()
fig.tight_layout()
os.makedirs(f"{OUT}/figures", exist_ok=True)
fig.savefig(f"{OUT}/figures/task_success.png", dpi=200)
plt.show()

done = sum(s["per_mode"][m]["n"] for m in order)
print(f"{done} of 30 runs finished ({s['failed_runs']} failed). To continue, save a new version with this one added as input.")

## 7. Output

The whole results folder is this version's output. This also makes a small zip of the report, figure and run summaries (without the large trajectories) to download from the Output tab.

In [ ]:
!cd {OUT} && zip -qr /kaggle/working/mk33_task_success_7b_summary.zip report.md summary.json curves.csv figures runs -x '*/workspace/*' '*/trajectory.json'
!ls -la /kaggle/working